# Notebook Integrador - IIA-Arvores-AsaNorte

Este notebook integra todas as fases do Projeto 2 de Introdução à Inteligência Artificial (CIC/UnB).
Ele combina os trabalhos individuais realizados nas branches dos membros da equipe e fornece mocks funcionais para as etapas subsequentes.

### 👥 Mapeamento de Responsabilidades:
- **Fase 1: Data Pipeline & Slicing** - Luidgi Varela (Pessoa 1) e Rafael Lima (Pessoa 2)
- **Fase 2: Pseudo-Labelling & QA** - Vitor Lopes (Pessoa 3), Felipe Costa (Pessoa 4), Lucas Saad (Pessoa 5) e Artur Kohara (Pessoa 6)
- **Fase 3: YOLO Training & Fine-Tuning** - Wallysson (Pessoa 7) e Célio Eduardo (Pessoa 8)
- **Fase 4: Avaliação & Relatório** - Arthur Botelho (Pessoa 9)

In [ ]:
import os
import sys
import cv2
import h5py
import numpy as np
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt

# Garantir que podemos importar de utils.py
sys.path.append('..')
import utils

## 🔵 Fase 1 - Data Pipeline & Slicing (Pessoa 1 - Luidgi Varela)

Esta seção baixa um recorte pequeno de ortofoto via WMS (opcional) e fatia a imagem GeoTIFF em tiles de 640x640.

In [ ]:
# Função de fatiamento espacial (Pessoa 1)
import rasterio
from rasterio.windows import Window

def slice_geotiff_local(input_path, output_dir, tile_size=640):
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    if not input_path.exists():
        raise FileNotFoundError(f"GeoTIFF não encontrado: {input_path}")
    
    output_dir.mkdir(parents=True, exist_ok=True)
    generated_tiles = []
    
    with rasterio.open(input_path) as src:
        meta = src.meta.copy()
        img_width = src.width
        img_height = src.height
        
        for row_off in range(0, img_height - tile_size + 1, tile_size):
            for col_off in range(0, img_width - tile_size + 1, tile_size):
                window = Window(col_off=col_off, row_off=row_off, width=tile_size, height=tile_size)
                tile_transform = rasterio.windows.transform(window, src.transform)
                
                tile_meta = meta.copy()
                tile_meta.update({
                    "height": tile_size,
                    "width": tile_size,
                    "transform": tile_transform
                })
                
                tile_name = f"{input_path.stem}_tile_{row_off}_{col_off}.tif"
                output_path = output_dir / tile_name
                
                with rasterio.open(output_path, "w", **tile_meta) as dst:
                    dst.write(src.read(window=window))
                generated_tiles.append(output_path)
    return generated_tiles

print("slice_geotiff carregado com sucesso!")

## 🔵 Fase 1 - Conversão Cromática e Criação do HDF5 (Pessoa 2 - Rafael Lima)

Esta seção converte os canais das imagens GeoTIFF para BGR, formata como JPEG e compila tudo no arquivo HDF5 bruto.

In [ ]:
# Conversão e compilação HDF5 bruto (Pessoa 2)
def executar_pipeline_hdf5(hdf5_path, tile_paths):
    print(f"Criando dataset HDF5 em: {hdf5_path} com {len(tile_paths)} tiles...")
    utils.criar_hdf5_bruto(hdf5_path, tile_paths)
    print("HDF5 Bruto criado com sucesso!")

# Exemplo de definição de caminhos
HDF5_RAW_PATH = "dataset_v1_raw.h5"
print("Funções do Rafael Lima prontas.")

## 🟡 Fase 2 - Geração de Pseudo-Labels com DeepForest (Pessoa 3 - Vitor Lopes)

Esta seção executa o modelo pré-treinado DeepForest (usando um mock para compatibilidade de ambiente) e salva as coordenadas YOLO no HDF5.

In [ ]:
# Pseudo-labelling com DeepForest (Pessoa 3)
class FakeDeepForest:
    def predict_image(self, image):
        # Simula a detecção de uma árvore no centro do tile
        return [
            {"xmin": 200, "ymin": 200, "xmax": 440, "ymax": 440, "score": 0.92}
        ]

def aplicar_pseudo_labels(hdf5_path):
    print(f"Aplicando pseudo-labels no HDF5: {hdf5_path}...")
    df_model = FakeDeepForest()
    with h5py.File(hdf5_path, "r+") as f:
        images = f["images"]
        for name in images.keys():
            img_data = images[name][:]
            detections = df_model.predict_image(img_data)
            
            boxes = []
            for det in detections:
                # Converter para YOLO normalizado [class_id, x_center, y_center, w, h]
                dw = 1.0 / 640.0
                dh = 1.0 / 640.0
                x = (det["xmin"] + det["xmax"]) / 2.0 * dw
                y = (det["ymin"] + det["ymax"]) / 2.0 * dh
                w = (det["xmax"] - det["xmin"]) * dw
                h = (det["ymax"] - det["ymin"]) * dh
                boxes.append([0.0, x, y, w, h])
            
            utils.salvar_pseudo_labels(hdf5_path, name, np.array(boxes, dtype=np.float32))
    print("Pseudo-labels gravadas no HDF5!")

print("Processo de Pseudo-Labelling pronto.")

## 🟡 Fase 2 - Exportação HDF5 para Curação (Pessoa 4 - Felipe Costa)

Extrai as imagens e as pseudo-labels em uma pasta local estruturada e gera um arquivo compactado `.zip` para upload no Roboflow.

In [ ]:
# Exportação para o Roboflow (Pessoa 4)
def exportar_para_curadoria(hdf5_path, output_dir):
    print(f"Exportando {hdf5_path} para Roboflow em {output_dir}...")
    res = utils.exportar_hdf5_para_roboflow(hdf5_path, output_dir)
    print(f"Exportação concluída! Zip gerado em: {res['zip_path']}")
    return res

ROBOFLOW_DIR = "roboflow_export"
print("Função de exportação pronta.")

## 🟡 Fase 2 - Curadoria Manual e Controle de Qualidade (Pessoa 5 - Lucas Saad) [MOCK]

Esta etapa é manual e ocorre na plataforma Roboflow. Simulamos o resultado limpando possíveis falsos positivos.

In [ ]:
# Simulação de curadoria e controle de qualidade (Pessoa 5)
def simular_curadoria(hdf5_in, hdf5_out):
    print(f"[MOCK] Simulando curadoria de {hdf5_in} -> {hdf5_out}...")
    # Cria uma cópia do arquivo HDF5 simulando o download do Roboflow limpo
    import shutil
    shutil.copyfile(hdf5_in, hdf5_out)
    print("[MOCK] Curadoria concluída e salva no Drive como dataset_limpo.h5!")

HDF5_CLEAN_PATH = "dataset_limpo.h5"
print("Simulação de Curadoria pronta.")

## 🟡 Fase 2 - Particionamento Geográfico Asa Norte/Sul (Pessoa 6 - Artur Kohara) [MOCK]

Esta seção divide o dataset em treino e validação de forma geográfica para evitar vazamento espacial de dados.

In [ ]:
# Particionamento geográfico (Pessoa 6)
def simular_particionamento(hdf5_clean, out_train_path, out_val_path):
    print(f"[MOCK] Particionando geograficamente {hdf5_clean}...")
    # Em produção, divide as chaves do HDF5 por sua coordenada espacial
    # Aqui simulamos criando os arquivos de treino e validação separando tiles
    with h5py.File(hdf5_clean, 'r') as f_in:
        tile_names = list(f_in['images'].keys())
        split_idx = int(len(tile_names) * 0.8) # 80% treino
        
        train_tiles = tile_names[:split_idx]
        val_tiles = tile_names[split_idx:]
        
        # Cria dataset de treino
        with h5py.File(out_train_path, 'w') as f_tr:
            g_img = f_tr.create_group('images')
            g_lbl = f_tr.create_group('labels')
            for name in train_tiles:
                g_img.create_dataset(name, data=f_in['images'][name][:])
                g_lbl.create_dataset(name, data=f_in['labels'][name][:])
                
        # Cria dataset de validação
        with h5py.File(out_val_path, 'w') as f_val:
            g_img = f_val.create_group('images')
            g_lbl = f_val.create_group('labels')
            for name in val_tiles:
                g_img.create_dataset(name, data=f_in['images'][name][:])
                g_lbl.create_dataset(name, data=f_in['labels'][name][:])
                
    print(f"[MOCK] Treino salvo em {out_train_path} ({len(train_tiles)} tiles)")
    print(f"[MOCK] Validação salva em {out_val_path} ({len(val_tiles)} tiles)")

HDF5_TRAIN_PATH = "dataset_treino.h5"
HDF5_VAL_PATH = "dataset_val.h5"
print("Particionamento Geográfico configurado.")

## 🟢 Fase 3 - Dataloader e RAM Disk (Pessoa 7 - Wallysson) [MOCK]

Esta seção lê os arquivos HDF5 particionados e extrai as imagens e labels na RAM do Kaggle (`/dev/shm` ou `/tmp`) no formato clássico do YOLO.

In [ ]:
import os
import gdown
import h5py
import cv2
import numpy as np

ID_TREINO = "1JVWdwf00AOTaUC5-m9xDJi_98j1spBgJ"
ID_VAL = "12vFJVEZcsfJCqwXu5OV1sHpGDo_PclKe"
ID_YAML = "1Xk8wNL-RaoCrwFkS_D6VILlr4DNm5JGC"

print("Baixando os arquivos oficiais do Drive...")
gdown.download(f'https://drive.google.com/uc?id={ID_TREINO}', 'dataset_treino.h5', quiet=False)
gdown.download(f'https://drive.google.com/uc?id={ID_VAL}', 'dataset_val.h5', quiet=False)

base_ram_dir = "/dev/shm/dataset"
os.makedirs(base_ram_dir, exist_ok=True)
gdown.download(f'https://drive.google.com/uc?id={ID_YAML}', f'{base_ram_dir}/data.yaml', quiet=False)


def extrair_hdf5_para_yolo(hdf5_path, output_dir):
    """Lê o arquivo HDF5 e extrai imagens (.jpg) e labels (.txt) para o diretório de destino."""
    print(f"Extraindo {hdf5_path} para {output_dir}...")
    
    img_dir = os.path.join(output_dir, "images")
    lbl_dir = os.path.join(output_dir, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    
    with h5py.File(hdf5_path, 'r') as f:
        images = f['images']
        labels = f['labels']
        
        for name in images.keys():
            # Salva a imagem
            img_data = images[name][:]
            img_path = os.path.join(img_dir, f"{name}.jpg")
            cv2.imwrite(img_path, img_data)
            
            # Salva o arquivo de texto com as coordenadas YOLO
            lbl_data = labels[name][:]
            lbl_path = os.path.join(lbl_dir, f"{name}.txt")
            with open(lbl_path, 'w') as txt_file:
                for box in lbl_data:
                    # Formato YOLO: class_id x_center y_center width height
                    linha = f"{int(box[0])} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n"
                    txt_file.write(linha)

# Executa a extração real para as pastas de treino e validação no /dev/shm
print(f"\nIniciando extração REAL para RAM Disk em: {base_ram_dir}")
extrair_hdf5_para_yolo('dataset_treino.h5', os.path.join(base_ram_dir, "train"))
extrair_hdf5_para_yolo('dataset_val.h5', os.path.join(base_ram_dir, "val"))

print("\n🚀 Fase 3 concluída! Dataset extraído com sucesso na RAM e pronto para o YOLO!")

## 🟢 Fase 3 - Treinamento do YOLOv11m (Pessoa 8 - Célio Eduardo) [MOCK]

Esta seção configura e executa o treinamento do YOLOv11m, aplicando congelamento de backbone e data augmentations apropriadas.

In [ ]:
# Configuração de treinamento do YOLOv11m (Pessoa 8)
def simular_treinamento_yolo(data_yaml_path):
    print(f"[MOCK] Inicializando treinamento YOLOv11m usando {data_yaml_path}...")
    print("[MOCK] Backbone do YOLOv11m congelado nas primeiras 10 camadas (freeze=10).")
    print("[MOCK] Aumentações de dados ativadas: flipud=0.5, fliplr=0.5, degrees=90.0, mosaic=1.0.")
    print("[MOCK] Iniciando épocas... (simulando convergência de perdas)")
    for epoch in range(1, 4):
        print(f"Epoch {epoch}/100: Box Loss = {0.8/epoch:.4f}, Class Loss = {0.5/epoch:.4f}, mAP50 = {0.4 + 0.15*epoch:.4f}")
    print("[MOCK] Treinamento finalizado. Melhores pesos salvos em 'best.pt'!")
    # Cria arquivo mock de pesos
    with open("best.pt", "w") as f:
        f.write("pesos_yolo_mock_data")

print("Módulos de treinamento prontos.")

## 🟢 Fase 4 - Avaliação e Métricas Finais (Pessoa 9 - Arthur Botelho) [MOCK]

Esta seção calcula as métricas de validação pós-treino (Matriz de Confusão, Curvas PR) e realiza inferências visuais.

In [ ]:
# Avaliação estatística final (Pessoa 9)
def simular_avaliacao_e_inferencias(pesos_path):
    print(f"[MOCK] Carregando pesos do modelo de: {pesos_path}...")
    print("[MOCK] Gerando métricas de validação...")
    
    # Plota curvas mock
    recalls = np.linspace(0.0, 1.0, 100)
    precisions = 1.0 - recalls**2 # Mock curve
    
    plt.figure(figsize=(6, 4))
    plt.plot(recalls, precisions, label='YOLOv11m (mAP@0.5 = 0.85)')
    plt.title('Curva Precisão-Revogação (MOCK)')
    plt.xlabel('Revogação (Recall)')
    plt.ylabel('Precisão (Precision)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    print("[MOCK] Matriz de Confusão:")
    print("                 Predec: Árvore   Predec: Fundo")
    print("Real: Árvore       1450             210")
    print("Real: Fundo          85            9800")
    print("\nAvaliação final concluída! Pronto para integração no relatório de 5 páginas.")

print("Módulo de Avaliação pronto.")

## 🚀 Execução Completa e Integrada do Pipeline

Para rodar todo o pipeline de forma integrada (com os mocks ativados), execute a célula abaixo.

In [ ]:
# Criação de diretórios temporários para teste local do integrador
import shutil
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/tiles").mkdir(parents=True, exist_ok=True)

# 1. Criar uma imagem GeoTIFF de teste se não houver
test_tiff = Path("data/raw/asa_norte_setor_01.tif")
if not test_tiff.exists():
    print("Criando GeoTIFF sintético para fins de teste local...")
    # Cria um GeoTIFF sintético pequeno (1280x1280, 3 canais)
    import mock_tiff_creator # Criador local temporário
    mock_tiff_creator.create_synthetic_tiff(str(test_tiff))

# Executa o fatiamento (Fase 1)
tiles = slice_geotiff_local(test_tiff, "data/tiles/asa_norte_setor_01", tile_size=640)

# Executa a conversão cromática e compila HDF5 bruto (Fase 1)
executar_pipeline_hdf5(HDF5_RAW_PATH, tiles)

# Executa a geração de pseudo-labels com DeepForest (Fase 2)
aplicar_pseudo_labels(HDF5_RAW_PATH)

# Exporta para curadoria no Roboflow (Fase 2)
exportar_para_curadoria(HDF5_RAW_PATH, ROBOFLOW_DIR)

# Simula curadoria visual (Fase 2)
simular_curadoria(HDF5_RAW_PATH, HDF5_CLEAN_PATH)

# Simula particionamento geográfico Asa Norte/Sul (Fase 2)
simular_particionamento(HDF5_CLEAN_PATH, HDF5_TRAIN_PATH, HDF5_VAL_PATH)

# Extração para o RAM disk simulado (Fase 3)
extrair_para_treinamento(HDF5_TRAIN_PATH, HDF5_VAL_PATH, RAM_DIR)

# Simulação de treino (Fase 3)
simular_treinamento_yolo(os.path.join(RAM_DIR, "data.yaml"))

# Simulação de avaliação estatística (Fase 4)
simular_avaliacao_e_inferencias("best.pt")

# Limpeza dos mocks locais gerados pelo teste integrador
for path in [HDF5_RAW_PATH, HDF5_CLEAN_PATH, HDF5_TRAIN_PATH, HDF5_VAL_PATH, "best.pt"]:
    if os.path.exists(path):
        os.remove(path)
if os.path.exists(ROBOFLOW_DIR):
    shutil.rmtree(ROBOFLOW_DIR)
if os.path.exists(RAM_DIR):
    shutil.rmtree(RAM_DIR)
if os.path.exists("mock_tiff_creator.py"):
    os.remove("mock_tiff_creator.py")
print("\n--- PIPELINE INTEGRADO MOCK EXECUTADO COM SUCESSO! ---")